# 008 — D/LIVE: бустинг на слабых метках вне публичного отрезка
Run All создаёт геометрические признаки, слабую разметку из событий, обучает HistGradientBoostingClassifier и проверяет D на публичных пяти минутах. Новых библиотек нет. H/A не обучаются и не предсказываются.

Публичный интервал 11185–18684 и защитный запас 20 секунд с обеих сторон исключены из обучения. Истинные метки открываются после обучения и прогноза. Параметры дерева и пороги слабой разметки фиксированы заранее, проверки не выбирают модель. Временная проверка на слабой разметке: первый тайм → второй, затем финальное обучение на всех доступных слабых метках вне публичного окна. Нет случайного разделения кадров и автоматического early stopping.

Положительные слабые метки: неподвижный мяч ≥1 с, медленные полевые игроки и подходящее положение мяча в участке от 12 до 0,5 с перед началом интервала стандарта; либо устойчивый выход мяча за поле рядом со стандартом. Для аута проверяется близость боковой, для углового — угла, удара от ворот — лицевой зоны, начального удара — центра. Отрицательные: непрерывно наблюдаемый движущийся мяч внутри поля и движение игроков вдали от стандартов. Неизвестные кадры не размечаются как LIVE для обучения.

Выборка ≤5 FPS и ≤50 кадров на непрерывную слабую серию. Веса выравнивают вклад серий и классов, учитывая условную надёжность правил. Число слабых серий не равно числу независимых настоящих остановок. Дерево может копировать эвристику учителя; отдельно сравниваем сами правила.

Чистка — как в 007: предполагаемые вратари и suspect исключены из групповой геометрии. Наличие вратаря и расстояние до него — отдельные признаки. Окна 1/3/5/10 с смотрят назад и сбрасываются на границе тайма. События, абсолютное время и номера кадров не являются признаками. Пропуски остаются NaN. Необходима исходная тактическая разметка ATD/XML и Smart Tagging.

Выходы local_review/experiments/008_weak_dead_boosting: model.pkl, model_config.json, binary_predictions.csv (D/LIVE/X, НЕ конкурсный submission), p_D.npy, dead_mask.npy, metrics.json, REVIEW_RU.md. Модель из pickle загружать только из собственного доверенного файла. Данные, признаки и выходы остаются локальными.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS","4")
from pathlib import Path
import sys, json, time, warnings, hashlib, itertools, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
sys.path.insert(0,str(ROOT))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from starter.load_smart_tagging import load as load_smart_tagging
from evaluation import possession_metrics as pm
EXPERIMENT = "008_weak_dead_boosting"
OUT = ROOT / "local_review/experiments" / EXPERIMENT
OUT.mkdir(parents=True,exist_ok=True)
ignore = ROOT / "local_review/.gitignore"
if not ignore.exists(): ignore.write_text("*\n")
CFG = json.loads((ROOT / "sol/001_box5_vote100/config.json").read_text())

N, FPS = GAMES[GAME]["n_video_frames"], GAMES[GAME]["fps"]
PITCH = np.array(GAMES[GAME]["pitch"],dtype=np.float64)
BLOCKS = half_video_blocks(GAME)
PATHS = asset_paths()

def save_json(path, value):
    temp = path.with_suffix(path.suffix+".tmp")
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False))
    temp.replace(path)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

started=time.perf_counter()
print(EXPERIMENT, "| Results:", OUT)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support,average_precision_score,confusion_matrix
import pickle
PUBLIC_START,PUBLIC_END=11185,18685
GUARD=20*FPS
PARAMS=dict(max_iter=150,learning_rate=.05,max_leaf_nodes=7,max_depth=3,
            min_samples_leaf=25,l2_regularization=2.,early_stopping=False,random_state=42)


## Исходные данные, контрольный старый D и очистка

In [ ]:
# Load only ATD positions; raw speed units are inconsistent in the source documentation.
# Читаем только координаты ATD: единицы готовой скорости неоднозначны.
track_teams = team_of_each_track(PATHS["atd_meta"])
players = np.full((N, len(track_teams), 2), np.nan, dtype=np.float64)
ball = np.full((N, 2), np.nan, dtype=np.float64)
normalized_players = np.full((N,len(track_teams),2),np.nan,dtype=np.float64)
read_count = 0
with open(PATHS["atd_raw"]) as handle:
    for i, line in enumerate(tqdm(handle,total=N,desc="Tracking / Трекинг")):
        frame_id, player_text, ball_text = line.strip().split(":")
        assert int(frame_id) == i + 1, "ATD/video alignment mismatch"
        player_values = np.fromstring(player_text.replace(";", ","), sep=",").reshape(-1, 3)
        assert len(player_values) == len(track_teams)
        normalized_players[i] = player_values[:, :2]
        players[i] = player_values[:, :2] * PITCH
        ball[i] = np.fromstring(ball_text, sep=",")[:2] * PITCH
        read_count += 1
assert read_count == N
ball_seen = np.isfinite(ball).all(axis=1)
players_seen = np.isfinite(players).all(axis=2)

# The window is an axis-aligned SQUARE, not a radius-2.5 m circle.
# Окно — КВАДРАТ вдоль осей поля, не круг радиусом 2.5 м.
def box_votes(player_xy, ball_xy, team_ids, width_m):
    inside = (np.isfinite(player_xy).all(axis=2)
              & np.isfinite(ball_xy).all(axis=1)[:, None]
              & (np.abs(player_xy - ball_xy[:, None, :]) <= width_m / 2).all(axis=2))
    home_count = (inside & (team_ids == 0)).sum(axis=1)
    away_count = (inside & (team_ids == 1)).sum(axis=1)
    # +1 means H, -1 means A, 0 means abstention for this frame.
    # +1 — H, -1 — A, 0 — нет голоса в этом кадре.
    votes = np.sign(home_count - away_count).astype(np.int8)
    return home_count, away_count, votes

home_count, away_count, frame_votes = box_votes(players, ball, track_teams, CFG["box_width_m"])
# A narrow synthetic check verifies corners, ties, and missing-ball handling.
# Проверяем на искусственных точках углы квадрата, ничью и отсутствие мяча.
test_players = np.array([[[2.5, 2.5], [2.6, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]])
test_ball = np.array([[0., 0.], [0., 0.], [np.nan, np.nan]])
assert box_votes(test_players, test_ball, np.array([0, 1]), 5)[2].tolist() == [1, 0, 0]

# A set-piece interval provides context only; its team tag is not used as a possession label.
# Интервал стандарта — только контекст; его Team не копируем в метку владения.
set_piece_context = np.zeros(N, dtype=bool)
for event in load_smart_tagging():
    if (event["code"] or "").strip().upper() == "SET PIECES":
        start = max(0, event["start_frame"])
        stop = min(N, event["end_frame"])
        set_piece_context[start:stop] = True
print("ATD and automatic set-piece context loaded; no ground truth accessed.")


In [ ]:
lag = CFG["motion_lag_frames"]
dt = lag / FPS
player_motion = np.full(N, np.nan)
ball_motion = np.full(N, np.nan)

# Calculate motion separately in each half, so halftime never creates a false displacement.
# Движение считаем отдельно внутри таймов, не соединяя их через перерыв.
for _, first, last in BLOCKS:
    stop = last + 1
    speed = np.linalg.norm(players[first + lag:stop] - players[first:stop - lag], axis=2) / dt
    speed[(speed > CFG["max_player_motion_mps"]) | ~np.isfinite(speed)] = np.nan
    enough = np.isfinite(speed).sum(axis=1) >= CFG["minimum_motion_tracks"]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="All-NaN slice encountered", category=RuntimeWarning)
        medians = np.nanmedian(speed, axis=1)
    medians[~enough] = np.nan
    player_motion[first + lag:stop] = medians
    seen = ball_seen[first:stop]
    cumulative = np.r_[0, np.cumsum(seen)]
    continuous = (cumulative[lag + 1:] - cumulative[:-(lag + 1)]) == lag + 1
    ball_speed = np.linalg.norm(ball[first + lag:stop] - ball[first:stop - lag], axis=1) / dt
    ball_speed[~continuous] = np.nan
    ball_motion[first + lag:stop] = ball_speed

outside = ball_seen & ((ball < -CFG["outside_margin_m"]) | (ball > PITCH + CFG["outside_margin_m"])).any(axis=1)
inside_field = ball_seen & (ball >= 0).all(axis=1) & (ball <= PITCH).all(axis=1)

def detect_dead_ball():
    dead = np.zeros(N, dtype=bool)
    # Diagnostic counts record triggers, not ground-truth stopping reasons.
    # Счётчики показывают срабатывания эвристик, а не истинные причины остановок.
    reasons = {"outside": 0, "stationary": 0, "missing_and_slow": 0}
    for _, first, last in BLOCKS:
        state = False
        outside_run = missing_run = stationary_run = slow_run = restart_ball_run = restart_players_run = 0
        for i in range(first, last + 1):
            outside_run = outside_run + 1 if outside[i] else 0
            missing_run = missing_run + 1 if not ball_seen[i] else 0
            stationary_run = stationary_run + 1 if ball_motion[i] <= CFG["stationary_ball_mps"] else 0
            slow_run = slow_run + 1 if player_motion[i] <= CFG["slow_players_mps"] else 0
            restart_ball_run = restart_ball_run + 1 if inside_field[i] and ball_motion[i] >= CFG["restart_ball_mps"] else 0
            restart_players_run = restart_players_run + 1 if player_motion[i] >= CFG["restart_players_mps"] else 0
            if not state:
                stationary_threshold = (CFG["set_piece_stationary_confirm_frames"] if set_piece_context[i]
                                        else CFG["stationary_ball_confirm_frames"])
                slow = slow_run >= CFG["slow_players_confirm_frames"]
                if outside_run >= CFG["outside_confirm_frames"]:
                    state = True
                    reasons["outside"] += 1
                elif slow and stationary_run >= stationary_threshold:
                    state = True
                    reasons["stationary"] += 1
                elif slow and missing_run >= CFG["missing_ball_confirm_frames"]:
                    state = True
                    reasons["missing_and_slow"] += 1
            elif not outside[i] and (restart_ball_run >= CFG["restart_ball_confirm_frames"]
                                    or restart_players_run >= CFG["restart_players_confirm_frames"]):
                state = False
                # Reset evidence so a previous stoppage cannot immediately trigger a new one.
                # Сбрасываем накопленные признаки предыдущей остановки.
                outside_run = missing_run = stationary_run = slow_run = 0
            dead[i] = state
    return dead, reasons

dead, dead_trigger_counts = detect_dead_ball()


In [ ]:
# Reproduce the exact existing cleanup rules from original normalized ATD coordinates.
# Воспроизводим прежнюю чистку по исходным нормированным координатам ATD.
teams=track_teams; LENGTH=float(PITCH[0]); SIDE_SECONDS=10
KICKOFFS={1:11185,2:113742}
# Infer defending side from team-wide medians, so one stray extreme cannot flip the sides.
# Определяем сторону по медиане всей команды, чтобы одиночный выброс не перевернул стороны.
sides = {}
side_diagnostics = {}
for half, first, last in BLOCKS:
    start = KICKOFFS[half]
    stop = min(start + SIDE_SECONDS * FPS, last + 1)
    centers = {}
    for team in [0, 1]:
        x = normalized_players[start:stop, teams == team, 0]
        valid_rows = np.isfinite(x).sum(axis=1) >= 2
        assert valid_rows.sum() >= FPS, "Insufficient observations to infer sides"
        centers[team] = float(np.median(np.nanmedian(x[valid_rows], axis=1)))
    assert abs(centers[0] - centers[1]) > 0.05, "Side assignment ambiguous"
    sides[half] = {0: "left", 1: "right"} if centers[0] < centers[1] else {0: "right", 1: "left"}
    side_diagnostics[half] = {"home_defends": sides[half][0], "away_defends": sides[half][1],
                             "evidence_frames": stop-start, "method": "median_team_position",
                             "team_median_gap": abs(centers[0]-centers[1])}

# Only longitudinal depth is checked, not the lateral penalty-area boundaries.
# Проверяем глубину от своих ворот, без ограничения по ширине штрафной.
KEEPER_MAX_DEPTH_M = 16.5

def keeper_and_line(x, visible, left, length=LENGTH):
    """Return per-row keeper flags and the deepest remaining defender line."""
    values = x.copy()
    fill = np.inf if left else -np.inf
    values[~visible] = fill
    candidate = np.argmin(values, axis=1) if left else np.argmax(values, axis=1)
    rows = np.arange(len(values))
    candidate_x = values[rows, candidate]
    depth = candidate_x * length if left else (1 - candidate_x) * length
    # Behind-goal outliers cannot become keepers; the boundary is inclusive.
    # Выбросы за воротами не становятся вратарями; граница 16,5 м включается.
    eligible = visible.any(axis=1) & (depth >= -1e-10) & (depth <= KEEPER_MAX_DEPTH_M + 1e-10)
    keeper = np.zeros_like(visible)
    keeper[rows[eligible], candidate[eligible]] = True
    # Without an eligible keeper, retain the deepest player as a defender.
    # Если вратаря нет в наблюдениях, крайний игрок остаётся защитником.
    values[keeper] = fill
    line = np.min(values, axis=1) if left else np.max(values, axis=1)
    line[(visible.sum(axis=1) - eligible.astype(int)) == 0] = np.nan
    return keeper, line

# Small synthetic cases cover both sides, boundary, missing and single observations.
# Искусственные примеры: обе стороны, граница, пропуски и одиночные наблюдения.
test_x = np.array([[.165, .4], [.2, .4], [.1, np.nan], [.3, np.nan],
                   [np.nan, np.nan], [-.01, .1]])
for left in (True, False):
    coords = test_x if left else 1 - test_x
    k, line = keeper_and_line(coords, np.isfinite(coords), left, length=100)
    assert np.array_equal(k.sum(axis=1), [1, 0, 1, 0, 0, 0])
    expected = np.array([.4, .2, np.nan, .3, np.nan, -.01])
    np.testing.assert_allclose(line, expected if left else 1-expected, equal_nan=True)

keepers = np.zeros_like(players_seen)
last_outfield = np.full((N, 2), np.nan)
keeper_diagnostics = {}
for half, first, last in BLOCKS:
    rows = np.arange(first, last + 1)
    for team in [0, 1]:
        slots = np.flatnonzero(teams == team)
        x = normalized_players[first:last + 1, slots, 0]
        visible = players_seen[first:last + 1, slots]
        left = sides[half][team] == "left"
        keeper, line = keeper_and_line(x, visible, left)
        keepers[np.ix_(rows, slots)] = keeper
        last_outfield[rows, team] = line
        depth = x * LENGTH if left else (1-x) * LENGTH
        assert np.all((depth[keeper] >= -1e-10) & (depth[keeper] <= KEEPER_MAX_DEPTH_M + 1e-10))
        assert np.all(keeper.sum(axis=1) <= 1)
        keeper_diagnostics[f"half_{half}_team_{team}"] = {
            "frames_with_keeper": int(keeper.any(axis=1).sum()),
            "frames_with_players_without_keeper": int((visible.any(axis=1) & ~keeper.any(axis=1)).sum())}

suspect = np.zeros_like(players_seen)
flag_valid = np.zeros_like(players_seen)
for half, first, last in BLOCKS:
    rows = np.arange(first, last + 1)
    for team in [0, 1]:
        slots = np.flatnonzero(teams == team)
        opponent = 1 - team
        line = last_outfield[first:last + 1, opponent, None]
        x = normalized_players[first:last + 1, slots, 0]
        valid = players_seen[first:last + 1, slots] & np.isfinite(line)
        # Strict geometric comparison: no ball-position or halfway-line correction.
        # Строго заданная эвристика: без поправки на мяч и среднюю линию.
        behind = x < line if sides[half][opponent] == "left" else x > line
        flag_valid[np.ix_(rows, slots)] = valid
        suspect[np.ix_(rows, slots)] = valid & behind

# Flags must refer to players_seen points only.
# Флаги ставятся только существующим наблюдениям.
assert not np.any(suspect & ~flag_valid)
assert not np.any(keepers & ~players_seen)
summary = {"keeper_max_depth_m": KEEPER_MAX_DEPTH_M, "keeper_region": "own_goal_to_penalty_line_longitudinal_only",
           "keeper_diagnostics": keeper_diagnostics, "frames": N, "side_inference": side_diagnostics,
           "observed_player_points": int(players_seen.sum()), "observed_ball_points": int(ball_seen.sum()),
           "flagged_player_points": int(suspect.sum()), "valid_flag_checks": int(flag_valid.sum()),
           "raw_points_deleted": 0, "possession_model_executed": False}
print(json.dumps(summary, indent=2))

# If the previous cleaned CSV exists, require EXACT equality of all player flags.
# При наличии прежнего CSV проверяем точное совпадение всех флагов игроков.
clean_csv=ROOT/'local_review/tracking_cleanup/001_keeper_line/tracking_keeper_line_flags.csv'
csv_verified=False
if clean_csv.exists():
    checked=0
    columns=['frame','object_type','track_slot','is_goalkeeper_proxy','suspect_by_rule']
    for chunk in tqdm(pd.read_csv(clean_csv,usecols=columns,chunksize=200000),desc='Verify old cleanup / Проверка старой чистки'):
        chunk=chunk[chunk.object_type=='player']
        rr=chunk.frame.to_numpy(dtype=int);ss=chunk.track_slot.to_numpy(dtype=int)-1
        assert np.array_equal(keepers[rr,ss],chunk.is_goalkeeper_proxy.to_numpy(dtype=bool))
        assert np.array_equal(suspect[rr,ss],chunk.suspect_by_rule.to_numpy(dtype=bool))
        checked+=len(chunk)
    assert checked==int(players_seen.sum())
    csv_verified=True
clean_seen=players_seen&~keepers&~suspect



## Признаки без событий и эталонных меток

In [ ]:
# No frame ID, match clock, event code, event distance or true labels enter the model.
# В признаки не входят номер кадра, часы матча, события, расстояние до событий или эталон.
features={}
def add(name,value):features[name]=np.asarray(value,dtype=np.float32)
add('ball_seen',ball_seen.astype(float))
add('ball_x',ball[:,0]/PITCH[0]);add('ball_y',ball[:,1]/PITCH[1])
add('ball_touchline_distance',np.minimum(ball[:,1],PITCH[1]-ball[:,1]))
add('ball_goal_line_distance',np.minimum(ball[:,0],PITCH[0]-ball[:,0]))
add('ball_corner_distance',np.hypot(features['ball_touchline_distance'],features['ball_goal_line_distance']))
add('ball_outside',outside.astype(float))
add('clean_player_count',clean_seen.sum(1))
add('excluded_player_count',(players_seen&~clean_seen).sum(1))
add('keeper_count',keepers.sum(1))
# The keeper is excluded from team geometry but retained as separate evidence.
# Вратарь исключён из геометрии команд, но его близость к мячу сохранена отдельно.
dist=np.linalg.norm(players-ball[:,None,:],axis=-1)
for name,mask in [('outfield',clean_seen),('keeper',keepers&players_seen)]:
    d=np.where(mask&ball_seen[:,None],dist,np.inf)
    nearest=d.min(1);nearest[~np.isfinite(nearest)]=np.nan
    add('nearest_'+name,nearest)
for radius in [2,5,10]:add('players_near_ball_'+str(radius),((dist<=radius)&clean_seen&ball_seen[:,None]).sum(1))
del dist
with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning)
    for team in [0,1]:
        mask=clean_seen&(track_teams==team)
        points=np.where(mask[...,None],players,np.nan)
        center=np.nanmedian(points,axis=1)
        add(f'team{team}_count',mask.sum(1))
        for axis in [0,1]:
            add(f'team{team}_center{axis}',center[:,axis])
            add(f'team{team}_spread{axis}',np.nanpercentile(points[:,:,axis],75,axis=1)-np.nanpercentile(points[:,:,axis],25,axis=1))
        add(f'team{team}_ball_distance',np.linalg.norm(center-ball,axis=1))
        del points

# Only uninterrupted observations yield ball speeds. Gaps remain NaN, not zero.
# Скорость мяча вычисляется только при непрерывных наблюдениях; пропуски остаются NaN.
for lag in [5,25,50]:
    speed=np.full(N,np.nan)
    for half,first,last in BLOCKS:
        stop=last+1
        v=np.linalg.norm(ball[first+lag:stop]-ball[first:stop-lag],axis=1)/(lag/FPS)
        cs=np.r_[0,np.cumsum(ball_seen[first:stop])]
        uninterrupted=cs[lag+1:]-cs[:-lag-1]==lag+1
        v[~uninterrupted|(v>60)]=np.nan
        speed[first+lag:stop]=v
    add('ball_speed_'+str(lag),speed)
# Point motion is filtered, since tracker slots can switch identities.
# Движение точек фильтруется: слоты трекера могут менять личность.
median_motion=np.full(N,np.nan);slow_fraction=np.full(N,np.nan);fast_fraction=np.full(N,np.nan)
lag=10
for half,first,last in BLOCKS:
    stop=last+1
    v=np.linalg.norm(players[first+lag:stop]-players[first:stop-lag],axis=-1)/(lag/FPS)
    ok=clean_seen[first+lag:stop]&clean_seen[first:stop-lag]&np.isfinite(v)&(v<=12)
    count=ok.sum(1);v[~ok]=np.nan
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        median=np.nanmedian(v,axis=1)
    median[count<4]=np.nan
    median_motion[first+lag:stop]=median
    slow_fraction[first+lag:stop]=np.where(count>=4,(v<.7).sum(1)/np.maximum(count,1),np.nan)
    fast_fraction[first+lag:stop]=np.where(count>=4,(v>2).sum(1)/np.maximum(count,1),np.nan)
add('median_player_speed',median_motion);add('slow_player_fraction',slow_fraction);add('fast_player_fraction',fast_fraction)

def run_length(condition):
    out=np.zeros(N,dtype=np.float32)
    for _,first,last in BLOCKS:
        ids=np.arange(last-first+1);c=condition[first:last+1]
        out[first:last+1]=(ids-np.maximum.accumulate(np.where(~c,ids,-1)))/FPS
    return np.minimum(out,10.)  # Bound history to 10 s / Ограничиваем историю 10 секундами
add('ball_missing_seconds',run_length(~ball_seen))
add('ball_stationary_seconds',run_length(features['ball_speed_5']<=.35))
add('ball_outside_seconds',run_length(outside))
# Rolling windows reset at halftime; they look only backwards.
# Скользящие окна смотрят назад и сбрасываются на границе тайма.
rolling_names=['ball_seen','ball_speed_25','median_player_speed','slow_player_fraction',
               'nearest_outfield','clean_player_count','ball_touchline_distance']
for seconds in tqdm([1,3,5,10],desc='Temporal features / Временные признаки'):
    for name in rolling_names:
        mean=np.full(N,np.nan);std=np.full(N,np.nan)
        for _,first,last in BLOCKS:
            roll=pd.Series(features[name][first:last+1]).rolling(seconds*FPS,min_periods=1)
            mean[first:last+1]=roll.mean();std[first:last+1]=roll.std(ddof=0)
        add(f'{name}_mean_{seconds}s',mean);add(f'{name}_std_{seconds}s',std)
feature_names=list(features)
X=np.column_stack([features[k] for k in feature_names])
X[~np.isfinite(X)]=np.nan
np.save(OUT/'features.npy',X)
save_json(OUT/'feature_names.json',feature_names)
assert not any('event' in k or 'frame' in k or 'time'==k for k in feature_names)
print('Feature count:',len(feature_names))


## Слабые обучающие примеры

In [ ]:
# Automatic event intervals are weak supervision, never the true D intervals.
# Автоматические события дают слабую разметку, а не истинные интервалы D.
events=load_smart_tagging()
event_near=np.zeros(N,bool);before_restart=np.zeros(N,bool);typed_location=np.zeros(N,bool)
for event in events:
    if (event['code'] or '').upper()!='SET PIECES':continue
    a,b=event['start_frame'],event['end_frame']
    lo=max(0,a-15*FPS);hi=min(N,b+3*FPS)
    event_near[lo:hi]=True
    # A conservative pre-event region ends before the annotated restart window.
    # Консервативный участок заканчивается до начала интервала возобновления.
    left=max(0,a-12*FPS);right=max(left,min(N,a-int(.5*FPS)))
    before_restart[left:right]=True
    kind=(event['tags'].get('Type') or '').lower()
    if kind=='throw-in':location=features['ball_touchline_distance']<=3
    elif kind=='corner kick':location=features['ball_corner_distance']<=6
    elif kind=='goal kick':location=features['ball_goal_line_distance']<=18
    elif kind.startswith('kick off'):location=np.linalg.norm(ball-PITCH/2,axis=1)<=8
    else:location=np.ones(N,bool)
    typed_location[left:right]|=location[left:right]
stationary=before_restart&typed_location&ball_seen&(features['ball_stationary_seconds']>=1)&(median_motion<=1.2)
out=event_near&(features['ball_outside_seconds']>=.12)
positive=stationary|out
negative=(~event_near)&inside_field&(features['ball_speed_25']>=2)&(features['ball_speed_25']<=35)&(median_motion>=.8)
negative&=~positive
weak_y=np.full(N,-1,dtype=np.int8);weak_y[positive]=1;weak_y[negative]=0
confidence=np.where(out,.9,np.where(stationary,.65,np.where(negative,.7,0.)))
graded=np.zeros(N,bool)
for _,first,last in BLOCKS:graded[first:last+1]=True
# The entire public window plus a 20-second guard is excluded from training.
# Всё публичное окно плюс защитные 20 секунд исключены из обучения.
eligible=graded.copy();eligible[max(0,PUBLIC_START-GUARD):min(N,PUBLIC_END+GUARD)]=False
eligible&=weak_y>=0
# Sample whole weak-label runs at <=5 FPS, cap 50 frames/run, equalise run influence.
# Выборка до 5 FPS, не более 50 кадров серии; длинные серии не доминируют по весу.
indices=[];weights=[];group_ids=[];gid=0;episode_counts={0:0,1:0}
for half,first,last in BLOCKS:
    values=np.where(eligible[first:last+1],weak_y[first:last+1],-1)
    starts=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1];ends=np.r_[starts[1:],len(values)]
    for a,b in zip(starts,ends):
        label=int(values[a])
        if label<0 or b-a<5:continue
        sample=np.arange(first+a,first+b,5)
        if len(sample)>50:sample=sample[np.linspace(0,len(sample)-1,50).astype(int)]
        indices.extend(sample.tolist());weights.extend((confidence[sample]/len(sample)).tolist())
        group_ids.extend([gid]*len(sample));gid+=1;episode_counts[label]+=1
indices=np.array(indices);weights=np.array(weights);group_ids=np.array(group_ids)
assert len(indices)>0 and (weak_y[indices]==1).sum()>=20 and (weak_y[indices]==0).sum()>=20,'Too few weak examples'
assert np.all((indices<PUBLIC_START-GUARD)|(indices>=PUBLIC_END+GUARD))
np.savez(OUT/'weak_training_selection.npz',frame=indices,y=weak_y[indices],weight=weights,group=group_ids)
save_json(OUT/'weak_label_summary.json',dict(positive_frames=int(positive.sum()),negative_frames=int(negative.sum()),
    sampled_frames=len(indices),sampled_D=int((weak_y[indices]==1).sum()),sampled_LIVE=int((weak_y[indices]==0).sum()),
    positive_runs=episode_counts[1],negative_runs=episode_counts[0],public_window_used_for_training=False))
print('Weak samples:',len(indices),'D/LIVE runs:',episode_counts)


## Обучение с временной проверкой слабых меток

In [ ]:
def balanced_weights(ids):
    w=weights[ids].copy();target=weak_y[indices[ids]]
    # Equal total class influence AFTER equalising run weights.
    # Выравниваем вклад классов ПОСЛЕ выравнивания весов серий.
    for label in [0,1]:
        mask=target==label;assert mask.any()
        w[mask]/=w[mask].sum()
    return w*len(w)/w.sum()

def binary_scores(truth,pred,prob=None):
    precision,recall,f1,support=precision_recall_fscore_support(truth,pred,labels=[0,1],zero_division=0)
    r=dict(D_precision=float(precision[1]),D_recall=float(recall[1]),D_F1=float(f1[1]),
           LIVE_F1=float(f1[0]),binary_macroF1=float(f1.mean()),true_D_frames=int(support[1]),predicted_D_frames=int(np.sum(pred==1)))
    if prob is not None:r['D_average_precision']=float(average_precision_score(truth,prob))
    return r

def fit_model(ids,stage):
    args=PARAMS.copy();args.pop('max_iter')
    model=HistGradientBoostingClassifier(**args,warm_start=True,max_iter=25)
    for trees in tqdm(range(25,PARAMS['max_iter']+1,25),desc=stage):
        model.set_params(max_iter=trees)
        model.fit(X[indices[ids]],weak_y[indices[ids]],sample_weight=balanced_weights(ids))
        save_json(OUT/'progress.json',dict(stage=stage,trees=trees,total_trees=PARAMS['max_iter']))
    return model
# A temporal weak-label check: first half trains, second half validates.
# Временная проверка слабых меток: обучение на первом тайме, проверка на втором.
half2=BLOCKS[1][1]
train_ids=np.flatnonzero(indices<half2);validation_ids=np.flatnonzero(indices>=half2)
assert set(group_ids[train_ids]).isdisjoint(set(group_ids[validation_ids]))
diagnostic=fit_model(train_ids,'Weak temporal check / Проверка слабых меток')
p=diagnostic.predict_proba(X[indices[validation_ids]])[:,1]
save_json(OUT/'weak_validation_metrics.json',binary_scores(weak_y[indices[validation_ids]],p>=.5,p))
# Fixed parameters; weak validation is not used to search thresholds or select trees.
# Параметры фиксированы; по слабой проверке не выбираем порог или число деревьев.
model=fit_model(np.arange(len(indices)),'Final boosting / Итоговый бустинг')
with (OUT/'model.pkl').open('wb') as f:pickle.dump(model,f)
save_json(OUT/'model_config.json',dict(parameters=PARAMS,feature_names=feature_names,
    threshold=.5,min_dead_frames=25,max_live_gap_frames=10,
    public_exclusion=[PUBLIC_START-GUARD,PUBLIC_END+GUARD],
    events_as_features=False,public_labels_used_for_training=False,
    versions={k:importlib.metadata.version(k) for k in ['numpy','pandas','scikit-learn']}))
probability=np.full(N,np.nan,dtype=np.float32)
for a in tqdm(range(0,N,10000),desc='Full D probability / Прогноз D'):
    b=min(a+10000,N);probability[a:b]=model.predict_proba(X[a:b])[:,1]
np.save(OUT/'p_D.npy',probability)


## Проверка на публичном эталоне: только после обучения

In [ ]:
def spans(values):
    if not len(values):return []
    start=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1]
    return list(zip(start,np.r_[start[1:],len(values)]))
def duration_filter(raw):
    joined=raw.copy()
    for a,b in spans(raw):
        if not raw[a] and a>0 and b<len(raw) and b-a<=10:joined[a:b]=True
    result=joined.copy()
    for a,b in spans(joined):
        if joined[a] and b-a<25:result[a:b]=False
    return result
assert not duration_filter(np.r_[np.zeros(30,bool),np.ones(10,bool),np.zeros(30,bool)]).any()
assert duration_filter(np.r_[np.ones(25,bool),np.zeros(10,bool),np.ones(25,bool)]).all()
raw_dead=(probability>=.5)&graded
filtered=np.zeros(N,bool)
for _,first,last in BLOCKS:
    filtered[first:last+1]=duration_filter(raw_dead[first:last+1])
    assert all(b-a>=25 for a,b in spans(filtered[first:last+1]) if filtered[first+a])
labels=np.where(graded,np.where(filtered,'D','LIVE'),'X')
pd.DataFrame({'frame':np.arange(N),'p_D':np.where(graded,probability,np.nan),'label':labels}).to_csv(OUT/'binary_predictions.csv',index=False)
np.save(OUT/'dead_mask.npy',filtered)
# Ground truth is opened ONLY AFTER all fitting and probability predictions are finished.
# Эталон открываем ТОЛЬКО ПОСЛЕ обучения и получения вероятностей.
gt=pd.read_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
assert np.array_equal(gt.frame.to_numpy(),np.arange(N))
public=gt.label.isin(['H','A','D']).to_numpy();assert public.sum()==7500
assert not public[indices].any()
target=gt.label.eq('D').to_numpy()
report={}
for name,mask in [('public_5min',public),('last_minute',public&(np.arange(N)>=PUBLIC_START+6000))]:
    report[name]={
      'boosting_raw':binary_scores(target[mask],raw_dead[mask],probability[mask]),
      'boosting_filtered':binary_scores(target[mask],filtered[mask]),
      'weak_rules_as_detector_unknown_is_LIVE':binary_scores(target[mask],positive[mask]),
      'old_D_heuristic':binary_scores(target[mask],dead[mask]),
      'weak_label_coverage':float(np.mean(weak_y[mask]>=0)),
      'true_D_runs':sum(target[mask][a] for a,b in spans(target[mask])),
      'predicted_D_runs':sum(filtered[mask][a] for a,b in spans(filtered[mask]))}
    # JSON native integers / Обычные целые числа для JSON
    report[name]['true_D_runs']=int(report[name]['true_D_runs'])
    report[name]['predicted_D_runs']=int(report[name]['predicted_D_runs'])
report['weak_public_audit']=dict(
    true_D_with_positive_weak_label=int((public&target&(weak_y==1)).sum()),
    true_D_with_negative_weak_label=int((public&target&(weak_y==0)).sum()),
    true_D_without_weak_label=int((public&target&(weak_y==-1)).sum()),
    true_LIVE_with_positive_weak_label=int((public&~target&(weak_y==1)).sum()),
    true_D_ball_missing=int((public&target&~ball_seen).sum()))
save_json(OUT/'metrics.json',report)
lines=['# 008 — бустинг D/LIVE на слабой разметке','',
'Модель не обучалась на публичных 5 минутах и в защитном окне 20 секунд вокруг них. Однако этот матч и публичные метрики ранее исследовались: это диагностика, не новый независимый тест.',
'Вход: геометрия и движение. События используются только для слабых обучающих меток. Проверка на слабых метках не является оценкой истинного качества.',
'', '| Метод, публичные 5 минут | D precision | D recall | D F1 | LIVE F1 | binary macroF1 |','|---|---:|---:|---:|---:|---:|']
for name,r in report['public_5min'].items():
    if isinstance(r,dict):lines.append(f"| {name} | {r['D_precision']:.4f} | {r['D_recall']:.4f} | {r['D_F1']:.4f} | {r['LIVE_F1']:.4f} | {r['binary_macroF1']:.4f} |")
lines+=['','Правила как детектор: неизвестные слабые метки трактуются как LIVE только для этой контрольной оценки. При обучении неизвестные исключены.',
'Сглаживание: заполнение LIVE-разрывов до 10 кадров между D, удаление D короче 25 кадров. Параметры не подбирались по публичному эталону.',
'Это только бинарная модель: H/A не назначены. spellF1 и трёхклассовый macroF1 для неё не рассчитываются.',
'Все данные и реконструируемые признаки остаются локальными.']
(OUT/'REVIEW_RU.md').write_text('\n'.join(lines)+'\n')
save_json(OUT/'progress.json',dict(status='completed',features=len(feature_names),seconds=time.perf_counter()-started,
    model_sha256=sha256(OUT/'model.pkl'),prediction_sha256=sha256(OUT/'binary_predictions.csv')))
print(json.dumps(report,indent=2))
